# Финальный отчёт: сквозная оценка ALPR-системы

Всё, что попадает в защиту диплома, собрано здесь:

1. **Метрики детектора** YOLO26n на val (mAP50, mAP50-95, precision, recall)
2. **Ablation препроцессинга** (ДЗ2, ДЗ3, ДЗ7, ДЗ8) — как каждый шаг влияет на CER
3. **Сравнение OCR**: EasyOCR vs собственная CharCNN
4. **Сравнение трекеров**: KCF vs CSRT (ДЗ10 стиль) — FPS, успех
5. **End-to-end accuracy** пайплайна на тестовом наборе
6. **Galerея ошибок** — разбор типовых провалов

In [ ]:
import sys, yaml
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch
sns.set_theme(style='whitegrid')

CFG = yaml.safe_load(open('../config.yaml', encoding='utf-8'))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Метрики детектора (из runs/detect/ua_plates_yolo26n)

In [ ]:
from ultralytics import YOLO

model = YOLO('../models/yolo26_plate.pt')
data_yaml = Path(CFG['yolo_dataset_dir']) / 'data.yaml'
m = model.val(data=str(data_yaml))
metrics = {
    'mAP50': float(m.box.map50),
    'mAP50-95': float(m.box.map),
    'precision': float(m.box.mp),
    'recall': float(m.box.mr),
}
pd.DataFrame([metrics]).round(4)

In [ ]:
# Графики кривых обучения
results_csv = Path('../runs/detect/ua_plates_yolo26n/results.csv')
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    df[['train/box_loss','val/box_loss']].plot(ax=axes[0]); axes[0].set_title('Box loss')
    df[['metrics/mAP50(B)', 'metrics/mAP50-95(B)']].plot(ax=axes[1]); axes[1].set_title('mAP')
    plt.tight_layout(); plt.show()

## 2. Ablation препроцессинга (перезапуск из 03_ocr_baseline.ipynb)

Загружаем сохранённые метрики в одну сводную таблицу.

In [ ]:
# Если 03_ocr_baseline.ipynb сохранил pickle — читаем. Иначе — запускаем инлайн.
import pickle
abl_path = Path('../data/ocr_crops/ablation.pkl')
if abl_path.exists():
    summary = pickle.load(open(abl_path, 'rb'))
    summary
else:
    print('Запустите 03_ocr_baseline.ipynb, чтобы получить ablation-таблицу.')
    summary = None
summary

## 3. EasyOCR vs собственная CharCNN

Если CharCNN обучена (`models/char_cnn.pt` существует), мерим её на валидации Zenodo,
и сравниваем character-level accuracy с EasyOCR, запущенным на тех же кропах.

In [ ]:
from src.char_cnn import CharCNN, CHAR_CLASSES, class_idx

cnn_ckpt = Path('../models/char_cnn.pt')
if cnn_ckpt.exists():
    ckpt = torch.load(cnn_ckpt, map_location=DEVICE, weights_only=False)
    cnn = CharCNN().to(DEVICE); cnn.load_state_dict(ckpt['state_dict']); cnn.eval()
    print('CharCNN loaded. Val accuracy уже измерена в 02b_train_ocr_cnn.ipynb.')
else:
    print('CharCNN пока не обучена — см. 02b_train_ocr_cnn.ipynb')

## 4. Сравнение трекеров KCF vs CSRT (ДЗ10)

Прогоняем оба трекера на тестовом видео и сравниваем FPS / success rate / mean IoU.

In [ ]:
from src.tracker import benchmark_trackers

video = Path('../data/test_video.mp4')
if video.exists():
    cap = cv2.VideoCapture(str(video))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok: break
        frames.append(frame)
    cap.release()
    if frames:
        # Инициализация — YOLO на первом кадре
        model = YOLO('../models/yolo26_plate.pt')
        res = model.predict(frames[0], conf=0.25, verbose=False)[0]
        if len(res.boxes):
            x1, y1, x2, y2 = map(int, res.boxes[0].xyxy[0].tolist())
            init_bbox = (x1, y1, x2-x1, y2-y1)
            stats = benchmark_trackers(frames, init_bbox)
            df = pd.DataFrame({k: {
                'fps': v.fps, 'success_rate': v.success_rate, 'mean_iou': v.mean_iou, 'frames': v.frames
            } for k, v in stats.items()}).T
            display(df.round(3))
        else:
            print('На первом кадре номер не найден.')
    else:
        print('Видео пустое.')
else:
    print('Нет ../data/test_video.mp4 — запишите 5–10 секунд проезжающих машин.')

## 5. End-to-end accuracy

На тестовом наборе помеченных кропов (`data/ocr_crops/labels.csv`) считаем:
- **Plate accuracy** — % полностью правильно распознанных
- **Character accuracy** — 1 − CER
- FPS всего пайплайна end-to-end

In [ ]:
import time
from src.pipeline import ALPRPipeline, PipelineConfig

labels = Path('../data/ocr_crops/labels.csv')
if labels.exists():
    df = pd.read_csv(labels)
    df = df[df['text'].astype(str).str.len() > 0]
    pipeline = ALPRPipeline(PipelineConfig())
    correct = 0; chars_correct = 0; chars_total = 0; times = []
    for _, row in df.iterrows():
        crop = cv2.cvtColor(cv2.imread(f'../data/ocr_crops/{row["file"]}'), cv2.COLOR_BGR2RGB)
        t0 = time.perf_counter()
        rect = pipeline.preprocess_crop(crop)
        _, pred = pipeline.recognize(rect)
        times.append(time.perf_counter() - t0)
        gt = ALPRPipeline.postprocess(str(row['text']))
        if pred == gt: correct += 1
        for c in gt:
            chars_total += 1
            if c in pred: chars_correct += 1
    print(f'Plate accuracy    : {100*correct/len(df):.1f}% ({correct}/{len(df)})')
    print(f'Character accuracy: {100*chars_correct/max(chars_total,1):.1f}%')
    print(f'Mean latency      : {1000*np.mean(times):.1f} ms/crop (OCR part only)')
else:
    print('Нет labels.csv — прогоните prepare_ocr_crops.py и впишите GT.')

## 6. Галерея успех/провал

Типовые случаи для отчёта:
- ✅ чистый фронтальный номер днём — работает
- ✅ наклонённый номер — помогает rectify (ДЗ7)
- ✅ пересвет/тень — помогает gray-world (ДЗ2)
- ❌ сильное размытие (motion blur) — unsharp помогает частично
- ❌ очень мелкий номер (<30×10 px) — YOLO теряет
- ❌ нестандартные форматы (мото, спец.техника) — нужно дополнить датасет

## Итог

Пайплайн объединяет **все ключевые техники из курса**:

| Стадия | Метод | ДЗ |
|--------|-------|----|
| Детекция | YOLO26n fine-tune | ДЗ13–15 (CNN) |
| Препроцессинг | Gray-world + Unsharp + Harris + Homography + Otsu | ДЗ2,3,6,7,8 |
| Распознавание | EasyOCR (любой формат) + собственная CNN | ДЗ13,14,15 |
| Real-time | Tracker (KCF/CSRT) | ДЗ10 |
| Анализ данных | EDA | ДЗ12 |
